# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata
print(f"{meta.name}: {meta.description}\n")
print("Dataset citation:", getattr(meta, 'citeAs', 'N/A'))
print("License:", getattr(meta, 'license', 'N/A'))
print("Published on:", getattr(meta, 'datePublished', 'N/A'))

## 2. Data Overview
Review available record sets, their fields, and their `@id`s.

The dataset may contain several record sets. Let's enumerate record sets, their `@id`s, and for each, the fields (columns) provided.

This step helps you select suitable record sets/fields for analysis. All references are by their `@id`.

In [ ]:
# List all record sets and their fields (referenced by @id)

if hasattr(meta, 'recordSet') and meta.recordSet:
    print(f"Found {len(meta.recordSet)} record sets:\n")
    for rs in meta.recordSet:
        print(f"- RecordSet name: {getattr(rs, 'name', '<no name>')} (@id: {rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs,'@id', '')})")
        if hasattr(rs, 'field'):
            print("  Fields:")
            for field in rs.field:
                fname = getattr(field, 'name', '<no name>')
                fid = field['@id'] if isinstance(field, dict) and '@id' in field else getattr(field, '@id', '')
                ftype = getattr(field, 'dataType', 'N/A')
                print(f"    - {fname} (@id: {fid}) type: {ftype}")
else:
    # Some Croissant schemas expose record sets via `ds.metadata.recordSet` as a list of plain dicts
    # If empty, try to infer from `dataset._data['recordSet']`
    print("No record sets found via .metadata.recordSet. Attempting fallback to raw data structure...")
    record_sets = dataset._data.get('recordSet', [])
    if not record_sets:
        # In some datasets, the key is 'cr:recordSet' due to context vocab
        record_sets = dataset._data.get('cr:recordSet', [])
    print(f"Found {len(record_sets)} record set(s) in the raw JSON-LD:\n")
    for rs in record_sets:
        rs_id = rs.get('@id')
        name = rs.get('name', '<no name>')
        print(f"- RecordSet: {name} (@id: {rs_id})")
        if rs.get('field'):
            print('  Fields:')
            for field in rs['field']:
                # field can be dict or @id string; try to resolve field dict
                if isinstance(field, dict):
                    fid = field.get('@id', '')
                    fname = field.get('name', '<no name>')
                else:
                    fid = field
                    fname = field
                print(f"    - {fname} (@id: {fid})")
    # Save list of record set ids for next steps
    record_set_ids = [rs.get('@id') for rs in record_sets]
else:
    # If .metadata.recordSet found, build record_set_ids from them
    try:
        record_set_ids = [rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', None) for rs in meta.recordSet]
    except Exception:
        record_set_ids = []

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above. If the dataset contains multiple record sets, you can list and extract all or select by their `@id`.

In [ ]:
# Extract data from each record set using @id

# Prepare record_set_ids from previous step if not already present
if 'record_set_ids' not in locals():
    # Try to fetch from dataset._data or explicitly set
    record_sets = dataset._data.get('recordSet', [])
    if not record_sets:
        record_sets = dataset._data.get('cr:recordSet', [])
    record_set_ids = [rs.get('@id') for rs in record_sets]

print(f"Loading records from these record set @ids: {record_set_ids}")

dataframes = {}
for rs_id in record_set_ids:
    print(f"Reading records from record set @id: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"  Loaded {len(df)} records with columns: {df.columns.tolist()}")
        else:
            print(f"  No records found for this record set.")
    except Exception as e:
        print(f"  Could not load records: {e}")
if dataframes:
    first_rs_id = list(dataframes)[0]
    print(f"Preview of first record set (@id: {first_rs_id}):")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific field values, normalizing numeric fields, or grouping/categorizing by field values. 

For this dataset, choose a numeric field and filter, normalize, and group as example. Replace `<numeric_field_id>` and `<group_field_id>` with the actual field `@id` names from step 2/3 as needed.

In [ ]:
import numpy as np
# Select the first DataFrame and inspect columns
if dataframes:
    df = list(dataframes.values())[0]
    print("Columns:\n", df.columns.tolist())
    # ---
    # Example: Select numeric field (adjust @id as needed, e.g. 'coverage_percent' or similar field @id)
    # For illustration, find a column with 'coverage' or 'abundance', else use any float/int column
    numeric_field = None
    preferred_candidates = [c for c in df.columns if 'coverage' in c.lower() or 'abundance' in c.lower() or 'count' in c.lower() or 'mw' in c.lower()]
    # Heuristic: choose among candidates, else fall back to numeric ones
    if preferred_candidates:
        numeric_field = preferred_candidates[0]
    else:
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
    print(f"Selected numeric field for analysis: {numeric_field}")

    # Filtering
    threshold = df[numeric_field].mean() if numeric_field else 0
    filtered_df = df[df[numeric_field] > threshold] if numeric_field else df
    print(f"Filtered records with {numeric_field} > {threshold} (mean value):")
    display(filtered_df.head())

    # Normalization
    if numeric_field:
        filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

    # Grouping - pick another likely categorical field (e.g. by description or sequence/Experiment)
    group_candidates = [c for c in df.columns if c != numeric_field and (df[c].dtype == object or pd.api.types.is_categorical_dtype(df[c]))]
    group_field = group_candidates[0] if group_candidates else None
    print(f"Grouping field candidate: {group_field}")
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped average of {numeric_field} by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between selected fields using `matplotlib` or `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure we have data and selected fields to plot
if dataframes:
    df = list(dataframes.values())[0]
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        field = numeric_fields[0]
        plt.figure(figsize=(8,4))
        sns.histplot(df[field].dropna(), bins=30, kde=True)
        plt.title(f'Distribution of {field}')
        plt.xlabel(field)
        plt.ylabel('Frequency')
        plt.show()
    # Optionally, scatter plot
    if len(numeric_fields) >= 2:
        plt.figure(figsize=(6,6))
        sns.scatterplot(x=df[numeric_fields[0]], y=df[numeric_fields[1]])
        plt.title(f'Scatter of {numeric_fields[0]} vs. {numeric_fields[1]}')
        plt.xlabel(numeric_fields[0])
        plt.ylabel(numeric_fields[1])
        plt.show()

## 6. Conclusion

In this notebook, you loaded the FAIR² dataset from its Croissant schema using `mlcroissant`, explored its structure by referencing entities by their `@id`, extracted tabular data for analysis, performed basic filtering and normalization, grouped by categorical fields, and visualized numeric distributions. This workflow can be extended for advanced bioinformatics or machine learning tasks as required.

Remember to always reference dataset entities (record sets, fields, columns, etc.) by their `@id` for reproducibility and consistency.